# Chronos-T5 Base (200M) — M6 Rounds 2–12 inference

**Purpose.** Generalise the successful Round 1 pilot
(`Chronos_Base_200M_round1.ipynb`) to the remaining M6 rounds. For every round
2–12 this notebook performs **raw** `amazon/chronos-t5-base` inference on the
Stage 3 rolling-origin context and saves the untouched sampled trajectories.

| Item | Value |
|---|---|
| Model | `amazon/chronos-t5-base` (200M parameters), loaded **once** |
| Rounds | 2 … 12 (11 rounds, processed sequentially) |
| Context | 512 weekday daily log returns ending on each round's forecast origin |
| Assets | 100 official M6 assets, official order |
| Horizon | 20 weekdays per round |
| Trajectories | 100 sampled paths per asset |
| Raw output per round | `(100 assets, 100 samples, 20 steps)` |
| Random seed | 42, reset at the start of every round |

**Chronos is univariate.** Each of the 100 assets is forecast as its own
independent series. Assets are passed to the model in batches of 10 purely for
GPU efficiency — batching never lets one asset's history inform another's
forecast, and it does not change asset ordering or output shape.

**No preprocessing happens here.** The contexts were produced in Stages 2 and 3
(shared weekday calendar, forward-filled closure days, DRE's official
zero-return treatment, log returns, 512-row slice). This notebook loads those
values and passes them to the model unchanged: no normalising, standardising,
smoothing, clipping, averaging or NaN-filling. OGN's genuine leading `NaN`s stay
`NaN` — Chronos masks missing values natively through its attention mask.

**DRE.** Raw Chronos output is preserved exactly as produced, including the
post-acquisition rounds (9–12) where the model may emit non-zero values. The
official M6 zero-return rule for DRE is applied later, during competition-specific
post-processing — **not** here, and PLD is not used.

**Scope.** This notebook stops at the raw sampled trajectories. It does **not**
sum the 20 returns, convert to four-week returns, rank assets, build M6 quintile
probabilities, load realised outcomes, or compute RPS.

**Run the sections in order, top to bottom.** Each round is saved to Drive as
soon as it finishes, so a failure in a later round cannot lose earlier work.

## 1. Install Chronos

Installs the stable `chronos-forecasting` package — the same package used by the
Round 1 pilot. Pretrained weights are **not** downloaded here; they are fetched
from the Hugging Face Hub in section 6 into the Colab runtime cache, never into
the research repository.

Colab may ask you to restart the runtime after installing. If it does, restart
and then re-run from this cell.

In [2]:
%pip install -q chronos-forecasting

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.6/80.6 kB 7.9 MB/s eta 0:00:00


## 2. Imports and experiment settings

The single settings block for the experiment. Every value matches the successful
Round 1 run. `SERIES_BATCH_SIZE` is the only value you should need to change if
the Colab GPU runs out of memory — it affects speed and memory only, never the
100 assets, the 100 trajectories, the 20-step horizon, or the asset ordering.

`ROUND_SCHEDULE` is the pre-specified official M6 table (from Stage 3's
`data/metadata/m6_round_schedule.csv`); it is used to *verify* each loaded
context, never to modify it. `EXPECTED_CONTEXT_SHA256` holds the SHA-256 digests
of the repository's Stage 3 context files, so the copies you place in Drive can
be proven byte-identical to the validated originals.

In [3]:
import hashlib
import traceback
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from chronos import ChronosPipeline

# --- Experiment settings (identical to the successful Round 1 run) ----------
MODEL_ID = "amazon/chronos-t5-base"
CONTEXT_LENGTH = 512
PREDICTION_LENGTH = 20
NUM_SAMPLES = 100
RANDOM_SEED = 42

# Assets per forward pass. Lower this if Colab reports a CUDA OOM error.
# It changes speed/memory ONLY - never the results' shape or ordering.
SERIES_BATCH_SIZE = 10

ROUNDS = range(2, 13)

# Fail loudly if a Drive context is not byte-identical to the Stage 3 file.
# Set to False only if you deliberately re-saved a context file.
STRICT_CONTEXT_HASH = True

# --- Official M6 round schedule (Stage 3, pre-specified) --------------------
# round -> context start, origin (= context end), forecast start, forecast end
ROUND_SCHEDULE = {
    2:  ("2020-04-16", "2022-04-01", "2022-04-04", "2022-04-29"),
    3:  ("2020-05-14", "2022-04-29", "2022-05-02", "2022-05-27"),
    4:  ("2020-06-11", "2022-05-27", "2022-05-30", "2022-06-24"),
    5:  ("2020-07-09", "2022-06-24", "2022-06-27", "2022-07-22"),
    6:  ("2020-08-06", "2022-07-22", "2022-07-25", "2022-08-19"),
    7:  ("2020-09-03", "2022-08-19", "2022-08-22", "2022-09-16"),
    8:  ("2020-10-01", "2022-09-16", "2022-09-19", "2022-10-14"),
    9:  ("2020-10-29", "2022-10-14", "2022-10-17", "2022-11-11"),
    10: ("2020-11-26", "2022-11-11", "2022-11-14", "2022-12-09"),
    11: ("2020-12-24", "2022-12-09", "2022-12-12", "2023-01-06"),
    12: ("2021-01-21", "2023-01-06", "2023-01-09", "2023-02-03"),
}

# SHA-256 of the repository's Stage 3 context files (computed locally).
EXPECTED_CONTEXT_SHA256 = {
    2:  "4b1854a641bb0229231d151301c8294c3f4cdf313dabfebff6d61e7a7d7e50fa",
    3:  "efef3a5bf1b1f4504761e6899d83cb26e62e097ad29e8363dcacc69c13b37798",
    4:  "541ce612db093729e3eee4be20f8e1a1aaa1a042cdd723cd119cd2f04effbe91",
    5:  "b9d4f19490cc9da3260ccce91739213a2936d3c0a238d6e29346a4d1a727421f",
    6:  "b3f3265fd0283e807033a330ff0679f0aaec0bea51aa3d7e67403a58cb3c2478",
    7:  "80846de8dacbd539012a01b0165a92a8681488e355bbffb8f2f06574827b6f4a",
    8:  "03e2441a200633b4ea0ed21184857ab8def4d8995feca82bdfafc2945e409e67",
    9:  "b9326f356a5feed2db3821125733c3ba2af4a988ad570be1e67fb8a590f4a4de",
    10: "9326663904ba2bc66a2f87975169d3e6c00371469a28f8cfd557ccc4aa9b19ce",
    11: "1c6ef423f44fc01e5df0e2c46dbe41b83bc24cd31c94d82a3371f2f91eeb611a",
    12: "d2ac16cd5c815a4a97e836986fd5e13098015af1885e31785dcd4349d40cbcdf",
}

# Genuine leading missing history, per Stage 3 (verified locally).
# CARR is complete from round 2 onward; OGN (listed May 2021) gains 20 valid
# returns per round, so its leading NaN block shrinks by 20 each round.
EXPECTED_LEADING_NAN = {r: {"OGN": 302 - 20 * (r - 1)} for r in ROUNDS}

# Official M6 asset order - used to verify the loaded contexts, not to reorder them.
OFFICIAL_ASSET_ORDER = [
    "ABBV", "ACN", "AEP", "AIZ", "ALLE", "AMAT", "AMP", "AMZN", "AVB", "AVY",
    "AXP", "BDX", "BF-B", "BMY", "BR", "CARR", "CDW", "CE", "CHTR", "CNC",
    "CNP", "COP", "CTAS", "CZR", "DG", "DPZ", "DRE", "DXC", "EWA", "EWC",
    "EWG", "EWH", "EWJ", "EWL", "EWQ", "EWT", "EWU", "EWY", "EWZ", "FTV",
    "GOOG", "GPC", "GSG", "HIG", "HIGH.L", "HST", "HYG", "IAU", "ICLN",
    "IEAA.L", "IEF", "IEFM.L", "IEMG", "IEUS", "IEVL.L", "IGF", "INDA",
    "IUMO.L", "IUVL.L", "IVV", "IWM", "IXN", "JPEA.L", "JPM", "KR", "LQD",
    "MCHI", "META", "MVEU.L", "OGN", "PG", "PPL", "PRU", "PYPL", "RE",
    "REET", "ROL", "ROST", "SEGA.L", "SHY", "SLV", "SPMV.L", "TLT", "UNH",
    "URI", "V", "VRSK", "VXX", "WRK", "XLB", "XLC", "XLE", "XLF", "XLI",
    "XLK", "XLP", "XLU", "XLV", "XLY", "XOM",
]
N_ASSETS = len(OFFICIAL_ASSET_ORDER)

print(f"chronos-forecasting {version('chronos-forecasting')} | torch {torch.__version__}")
print(f"Experiment: {MODEL_ID} | rounds {min(ROUNDS)}-{max(ROUNDS)} ({len(list(ROUNDS))} rounds)")
print(f"Target raw output shape per round: ({N_ASSETS}, {NUM_SAMPLES}, {PREDICTION_LENGTH})")

chronos-forecasting 2.3.1 | torch 2.11.0+cu128
Experiment: amazon/chronos-t5-base | rounds 2-12 (11 rounds)
Target raw output shape per round: (100, 100, 20)


## 3. Mount Google Drive

Colab cannot see the local VS Code repository, so Google Drive supplies the
round context files and stores the outputs permanently (Colab's own disk is
wiped when the runtime ends).

Running this cell opens a Google authorisation prompt.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 4. Configure the Drive context and output folders

**Edit the two paths below — they are the only paths you need to change.**

* `DRIVE_CONTEXT_DIR` — one folder holding *all* the context CSVs
  (`round_01_context.csv` … `round_12_context.csv`) copied from the repository's
  `Data/processed/rolling_origins/`. Only rounds 2–12 are read here; Round 1 may
  sit in the same folder and is simply ignored.
* `DRIVE_OUTPUT_DIR` — where the 11 NPZ files and the combined report are
  written. It is created automatically.

Per-round paths are built automatically from the round number, so you never type
individual filenames.

In [5]:
# >>> EDIT THESE TWO LINES <<<
DRIVE_CONTEXT_DIR = Path("/content/drive/MyDrive/HonoursResearch/Round_1_Context")
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/HonoursResearch/outputs/chronos_t5_base")


def context_path(round_number: int) -> Path:
    return DRIVE_CONTEXT_DIR / f"round_{round_number:02d}_context.csv"


def samples_path(round_number: int) -> Path:
    return DRIVE_OUTPUT_DIR / f"chronos_t5_base_round{round_number:02d}_samples.npz"


REPORT_PATH = DRIVE_OUTPUT_DIR / "chronos_t5_base_rounds02_12_inference_report.md"

if not DRIVE_CONTEXT_DIR.is_dir():
    raise FileNotFoundError(
        f"Context folder not found:\n  {DRIVE_CONTEXT_DIR}\n"
        "Correct DRIVE_CONTEXT_DIR above (it must be the folder containing "
        "round_01_context.csv .. round_12_context.csv)."
    )

DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Context folder: {DRIVE_CONTEXT_DIR}")
print(f"Output folder : {DRIVE_OUTPUT_DIR}")
print(f"Report will be: {REPORT_PATH.name}")

Context folder: /content/drive/MyDrive/HonoursResearch/Round_1_Context
Output folder : /content/drive/MyDrive/HonoursResearch/outputs/chronos_t5_base
Report will be: chronos_t5_base_rounds02_12_inference_report.md


## 5. Check the Round 2–12 context files

Confirms all 11 files are present *before* the model is loaded, and verifies each
one is byte-identical to the Stage 3 file validated in the repository. Doing this
up front means the long inference loop cannot fail halfway because of a missing
or altered input.

In [6]:
missing, mismatched = [], []
CONTEXT_SHA256 = {}

for r in ROUNDS:
    path = context_path(r)
    if not path.is_file():
        missing.append(path.name)
        continue
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    CONTEXT_SHA256[r] = digest
    status = "match" if digest == EXPECTED_CONTEXT_SHA256[r] else "DIFFERS"
    if status == "DIFFERS":
        mismatched.append(r)
    print(f"  round {r:02d}: {path.name}  sha256 {digest[:16]}...  ({status})")

if missing:
    raise FileNotFoundError(
        "Missing context files in "
        f"{DRIVE_CONTEXT_DIR}:\n  " + "\n  ".join(missing)
    )
if mismatched and STRICT_CONTEXT_HASH:
    raise ValueError(
        f"Contexts for round(s) {mismatched} are not byte-identical to the "
        "Stage 3 files in the repository. Re-copy them from "
        "Data/processed/rolling_origins/ without opening or re-saving them."
    )

print(f"\nAll {len(CONTEXT_SHA256)} context files present and verified.")

  round 02: round_02_context.csv  sha256 4b1854a641bb0229...  (match)
  round 03: round_03_context.csv  sha256 efef3a5bf1b1f450...  (match)
  round 04: round_04_context.csv  sha256 541ce612db093729...  (match)
  round 05: round_05_context.csv  sha256 b9d4f19490cc9da3...  (match)
  round 06: round_06_context.csv  sha256 b3f3265fd0283e80...  (match)
  round 07: round_07_context.csv  sha256 80846de8dacbd539...  (match)
  round 08: round_08_context.csv  sha256 03e2441a200633b4...  (match)
  round 09: round_09_context.csv  sha256 b9326f356a5feed2...  (match)
  round 10: round_10_context.csv  sha256 9326663904ba2bc6...  (match)
  round 11: round_11_context.csv  sha256 1c6ef423f44fc01e...  (match)
  round 12: round_12_context.csv  sha256 d2ac16cd5c815a4a...  (match)

All 11 context files present and verified.


## 6. GPU check and model loading — **once**

> ⚠️ **Running this cell downloads/loads the pretrained Chronos model and places
> it on the Colab GPU.** The first run fetches roughly 200M parameters from the
> Hugging Face Hub into the runtime cache (not into your repository).

A GPU is required: if none is attached the cell stops with instructions rather
than silently running the 200M model on CPU. The model is loaded **once** here
and reused for all 11 rounds — it is never reloaded inside the loop. Inference
only: no training, no fine-tuning.

In [7]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. In Colab choose Runtime > Change runtime type > "
        "Hardware accelerator: GPU (T4 or better), then re-run this notebook "
        "from section 2. Refusing to run the 200M model on CPU."
    )

DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

pipeline = ChronosPipeline.from_pretrained(
    MODEL_ID,
    device_map="cuda",
    torch_dtype=DTYPE,
)

GPU_NAME = torch.cuda.get_device_name(0)
MODEL_LOADED_ONCE = True

print(f"GPU   : {GPU_NAME}")
print(f"dtype : {DTYPE}")
print(f"Loaded: {MODEL_ID} (inference only, no fine-tuning)")
print("Model loaded ONCE - it is reused for every round below.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  806MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

GPU   : NVIDIA L4
dtype : torch.bfloat16
Loaded: amazon/chronos-t5-base (inference only, no fine-tuning)
Model loaded ONCE - it is reused for every round below.


## 7. Rounds 2–12

### 7a. Per-round helpers

Three small functions, each doing exactly what the Round 1 notebook did for a
single round:

* `load_and_validate_context` — reads the CSV and asserts 512 rows, `date` plus
  the 100 official assets in order, ascending unique dates, the correct context
  start, an end equal to the round's official origin with nothing after it, and
  the expected genuine leading `NaN`s (no interior gaps). It then builds the
  `(100 assets, 512 steps)` model input and proves it is a plain transpose of the
  loaded values — nothing rescaled, filled or otherwise altered.
* `run_round_inference` — resets the seed to 42, forecasts the assets in batches
  of 10 as independent univariate series using the current Chronos API
  (`inputs=`), concatenates the batches back into official asset order, and
  asserts the `(100, 100, 20)` shape and finiteness.
* `save_and_verify_round` — writes the NPZ, then reloads it and checks that it
  opens, keeps the shape, keeps the asset order, carries the right forecast
  dates, and holds values identical to the in-memory array.

In [8]:
def load_and_validate_context(round_number: int):
    """Load one Stage 3 context, validate it, and build the model input.

    Returns (context_matrix, context_df, forecast_dates). No preprocessing of
    any kind is applied to the returns.
    """
    ctx_start, origin, fc_start, fc_end = ROUND_SCHEDULE[round_number]
    df = pd.read_csv(context_path(round_number), parse_dates=["date"])

    # --- Structural validation ---------------------------------------------
    assert df.shape[0] == CONTEXT_LENGTH, (
        f"Round {round_number}: expected {CONTEXT_LENGTH} rows, found {df.shape[0]}"
    )
    assert list(df.columns) == ["date"] + OFFICIAL_ASSET_ORDER, (
        f"Round {round_number}: columns are not 'date' + the official M6 asset order"
    )
    dates = df["date"]
    assert dates.is_monotonic_increasing, f"Round {round_number}: dates not ascending"
    assert not dates.duplicated().any(), f"Round {round_number}: duplicate dates"
    assert dates.iloc[0] == pd.Timestamp(ctx_start), (
        f"Round {round_number}: context starts {dates.iloc[0].date()}, expected {ctx_start}"
    )
    assert dates.iloc[-1] == pd.Timestamp(origin), (
        f"Round {round_number}: context ends {dates.iloc[-1].date()}, expected origin {origin}"
    )
    assert (dates <= pd.Timestamp(origin)).all(), (
        f"Round {round_number}: context contains a date after the origin"
    )

    # --- Model input: (100 assets, 512 time steps), values untouched --------
    matrix = df[OFFICIAL_ASSET_ORDER].to_numpy(dtype=np.float64).T
    assert matrix.shape == (N_ASSETS, CONTEXT_LENGTH)
    assert np.array_equal(
        matrix, df[OFFICIAL_ASSET_ORDER].to_numpy().T, equal_nan=True
    ), f"Round {round_number}: model input differs from the loaded context values"

    # --- Genuine leading missing history (must remain NaN) -----------------
    leading_missing, interior_missing = {}, {}
    for i, symbol in enumerate(OFFICIAL_ASSET_ORDER):
        row = matrix[i]
        n_lead = int(np.argmax(~np.isnan(row))) if np.isnan(row[0]) else 0
        if n_lead:
            leading_missing[symbol] = n_lead
        if np.isnan(row[n_lead:]).any():
            interior_missing[symbol] = int(np.isnan(row[n_lead:]).sum())
    assert not interior_missing, (
        f"Round {round_number}: unexpected interior NaNs {interior_missing}"
    )
    assert leading_missing == EXPECTED_LEADING_NAN[round_number], (
        f"Round {round_number}: leading NaN pattern changed - "
        f"expected {EXPECTED_LEADING_NAN[round_number]}, found {leading_missing}"
    )

    # --- Forecast dates -----------------------------------------------------
    forecast_dates = pd.bdate_range(fc_start, fc_end)
    assert len(forecast_dates) == PREDICTION_LENGTH, (
        f"Round {round_number}: forecast window is {len(forecast_dates)} weekdays, "
        f"expected {PREDICTION_LENGTH}"
    )

    return matrix, df, forecast_dates


def run_round_inference(context_matrix, round_number: int):
    """Forecast the 100 assets independently and return the raw (100,100,20) array."""
    # Reset the seed per round so any single round can be rerun reproducibly.
    torch.manual_seed(RANDOM_SEED)
    torch.cuda.manual_seed_all(RANDOM_SEED)

    batch_outputs = []
    for start in range(0, N_ASSETS, SERIES_BATCH_SIZE):
        stop = min(start + SERIES_BATCH_SIZE, N_ASSETS)
        # One independent 1-D series per asset in this batch.
        batch_context = [
            torch.tensor(context_matrix[i], dtype=torch.float32)
            for i in range(start, stop)
        ]
        samples = pipeline.predict(
            inputs=batch_context,
            prediction_length=PREDICTION_LENGTH,
            num_samples=NUM_SAMPLES,
        )
        batch_outputs.append(samples.to(torch.float32).cpu().numpy())
        print(f"    assets {start + 1:3d}-{stop:3d} -> {tuple(batch_outputs[-1].shape)}")

    forecast_samples = np.concatenate(batch_outputs, axis=0)

    assert forecast_samples.shape == (N_ASSETS, NUM_SAMPLES, PREDICTION_LENGTH), (
        f"Round {round_number}: expected "
        f"({N_ASSETS}, {NUM_SAMPLES}, {PREDICTION_LENGTH}), got {forecast_samples.shape}"
    )
    assert np.isfinite(forecast_samples).all(), (
        f"Round {round_number}: forecasts contain NaN or infinite values"
    )
    return forecast_samples


def save_and_verify_round(round_number: int, forecast_samples, forecast_dates):
    """Save the raw array to Drive, then reload it and verify it."""
    path = samples_path(round_number)
    date_strings = np.array([d.strftime("%Y-%m-%d") for d in forecast_dates])

    np.savez_compressed(
        path,
        forecast_samples=forecast_samples,
        asset_symbols=np.array(OFFICIAL_ASSET_ORDER),
        forecast_dates=date_strings,
    )

    with np.load(path, allow_pickle=False) as reloaded:
        r_samples = reloaded["forecast_samples"]
        r_symbols = reloaded["asset_symbols"]
        r_dates = reloaded["forecast_dates"]

    assert r_samples.shape == (N_ASSETS, NUM_SAMPLES, PREDICTION_LENGTH), (
        f"Round {round_number}: reloaded shape {r_samples.shape}"
    )
    assert list(r_symbols) == OFFICIAL_ASSET_ORDER, (
        f"Round {round_number}: asset ordering changed on save/reload"
    )
    assert np.array_equal(r_samples, forecast_samples), (
        f"Round {round_number}: saved values differ from the forecasts"
    )
    assert np.isfinite(r_samples).all(), f"Round {round_number}: reloaded values not finite"
    assert list(r_dates) == list(date_strings), (
        f"Round {round_number}: forecast dates changed on save/reload"
    )
    return path


print("Helpers defined.")

Helpers defined.


### 7b. Run Rounds 2–12 sequentially

For each round in turn: load and validate the context → build the Chronos input
→ run inference → validate the output → **save immediately** and verify the saved
file. Only then does the next round start, so a failure at any point leaves every
previously completed round safely on Drive.

If a round raises, it is recorded and the loop continues with the next round; the
failures are listed in the summary so you can rerun just those rounds. Expect a
few minutes per round on a T4/L4.

In [9]:
round_records = []
failed_rounds = []
run_started_utc = datetime.now(timezone.utc)

for r in ROUNDS:
    ctx_start, origin, fc_start, fc_end = ROUND_SCHEDULE[r]
    print(f"\n=== Round {r:02d} | origin {origin} | forecast {fc_start} .. {fc_end} ===")
    try:
        context_matrix, context_df, forecast_dates = load_and_validate_context(r)
        print(f"  context validated: {context_matrix.shape} (assets x time steps), "
              f"{context_df['date'].iloc[0].date()} .. {context_df['date'].iloc[-1].date()}")

        started = datetime.now(timezone.utc)
        forecast_samples = run_round_inference(context_matrix, r)
        elapsed = (datetime.now(timezone.utc) - started).total_seconds()

        all_finite = bool(np.isfinite(forecast_samples).all())
        saved_path = save_and_verify_round(r, forecast_samples, forecast_dates)

        round_records.append({
            "round": r,
            "context_start": str(context_df["date"].iloc[0].date()),
            "origin": str(context_df["date"].iloc[-1].date()),
            "forecast_start": str(forecast_dates[0].date()),
            "forecast_end": str(forecast_dates[-1].date()),
            "shape": tuple(forecast_samples.shape),
            "all_finite": all_finite,
            "npz_filename": saved_path.name,
            "context_sha256": CONTEXT_SHA256[r],
            "seconds": round(elapsed, 1),
        })
        print(f"  raw output {tuple(forecast_samples.shape)} | all finite: {all_finite} "
              f"| {elapsed:.1f}s")
        print(f"  saved and verified -> {saved_path.name}")

    except Exception as exc:  # keep earlier rounds safe, report and continue
        failed_rounds.append((r, f"{type(exc).__name__}: {exc}"))
        print(f"  !! ROUND {r:02d} FAILED - earlier rounds remain saved")
        traceback.print_exc()

print(f"\nCompleted rounds: {[rec['round'] for rec in round_records]}")
if failed_rounds:
    print(f"FAILED rounds   : {[r for r, _ in failed_rounds]} (rerun these)")
else:
    print("No failures.")


=== Round 02 | origin 2022-04-01 | forecast 2022-04-04 .. 2022-04-29 ===
  context validated: (100, 512) (assets x time steps), 2020-04-16 .. 2022-04-01
    assets   1- 10 -> (10, 100, 20)
    assets  11- 20 -> (10, 100, 20)
    assets  21- 30 -> (10, 100, 20)
    assets  31- 40 -> (10, 100, 20)
    assets  41- 50 -> (10, 100, 20)
    assets  51- 60 -> (10, 100, 20)
    assets  61- 70 -> (10, 100, 20)
    assets  71- 80 -> (10, 100, 20)
    assets  81- 90 -> (10, 100, 20)
    assets  91-100 -> (10, 100, 20)
  raw output (100, 100, 20) | all finite: True | 27.9s
  saved and verified -> chronos_t5_base_round02_samples.npz

=== Round 03 | origin 2022-04-29 | forecast 2022-05-02 .. 2022-05-27 ===
  context validated: (100, 512) (assets x time steps), 2020-05-14 .. 2022-04-29
    assets   1- 10 -> (10, 100, 20)
    assets  11- 20 -> (10, 100, 20)
    assets  21- 30 -> (10, 100, 20)
    assets  31- 40 -> (10, 100, 20)
    assets  41- 50 -> (10, 100, 20)
    assets  51- 60 -> (10, 100, 20)
 

## 8. Verify every saved Round 2–12 output

An independent pass over the files actually on Drive — nothing here relies on the
in-memory arrays. Each NPZ is reopened and checked for the three expected arrays,
the `(100, 100, 20)` shape, the official asset ordering, the correct forecast
dates, and finite values.

In [10]:
verification_rows = []
ALL_ROUNDS_VERIFIED = True

for r in ROUNDS:
    path = samples_path(r)
    if not path.is_file():
        verification_rows.append((r, path.name, "MISSING", "-", "-", "-"))
        ALL_ROUNDS_VERIFIED = False
        continue

    _, _, fc_start, fc_end = ROUND_SCHEDULE[r]
    expected_dates = [d.strftime("%Y-%m-%d") for d in pd.bdate_range(fc_start, fc_end)]
    with np.load(path, allow_pickle=False) as f:
        keys = set(f.files)
        samples = f["forecast_samples"]
        symbols = list(f["asset_symbols"])
        fdates = list(f["forecast_dates"])

    ok_keys = keys == {"forecast_samples", "asset_symbols", "forecast_dates"}
    ok_shape = samples.shape == (N_ASSETS, NUM_SAMPLES, PREDICTION_LENGTH)
    ok_order = symbols == OFFICIAL_ASSET_ORDER
    ok_dates = fdates == expected_dates
    ok_finite = bool(np.isfinite(samples).all())
    if not all([ok_keys, ok_shape, ok_order, ok_dates, ok_finite]):
        ALL_ROUNDS_VERIFIED = False

    verification_rows.append((
        r, path.name,
        "OK" if all([ok_keys, ok_shape, ok_order, ok_dates, ok_finite]) else "FAILED",
        str(samples.shape),
        "yes" if ok_order else "NO",
        "yes" if ok_finite else "NO",
    ))

print(f"{'Rnd':>3}  {'file':<44} {'status':<7} {'shape':<16} {'order':<6} finite")
for row in verification_rows:
    print(f"{row[0]:>3}  {row[1]:<44} {row[2]:<7} {row[3]:<16} {row[4]:<6} {row[5]}")

print(f"\nAll rounds 2-12 verified on Drive: {ALL_ROUNDS_VERIFIED}")

Rnd  file                                         status  shape            order  finite
  2  chronos_t5_base_round02_samples.npz          OK      (100, 100, 20)   yes    yes
  3  chronos_t5_base_round03_samples.npz          OK      (100, 100, 20)   yes    yes
  4  chronos_t5_base_round04_samples.npz          OK      (100, 100, 20)   yes    yes
  5  chronos_t5_base_round05_samples.npz          OK      (100, 100, 20)   yes    yes
  6  chronos_t5_base_round06_samples.npz          OK      (100, 100, 20)   yes    yes
  7  chronos_t5_base_round07_samples.npz          OK      (100, 100, 20)   yes    yes
  8  chronos_t5_base_round08_samples.npz          OK      (100, 100, 20)   yes    yes
  9  chronos_t5_base_round09_samples.npz          OK      (100, 100, 20)   yes    yes
 10  chronos_t5_base_round10_samples.npz          OK      (100, 100, 20)   yes    yes
 11  chronos_t5_base_round11_samples.npz          OK      (100, 100, 20)   yes    yes
 12  chronos_t5_base_round12_samples.npz          O

## 9. Combined inference report

One report for all 11 rounds (not one per round), written to the Drive output
folder as `chronos_t5_base_rounds02_12_inference_report.md`. Copy it into the
repository as
`Results/Chronos_T5_Base_200M/reports/chronos_t5_base_rounds02_12_inference_report.md`
after the run.

In [11]:
if not round_records:
    raise RuntimeError("No rounds completed - nothing to report. Fix the failures above.")

table = "\n".join(
    f"| {rec['round']:02d} | {rec['origin']} | {rec['forecast_start']} .. {rec['forecast_end']} "
    f"| {rec['shape'][0]} x {rec['shape'][1]} x {rec['shape'][2]} "
    f"| {'all finite' if rec['all_finite'] else 'FAILED'} | `{rec['npz_filename']}` |"
    for rec in round_records
)

failure_note = (
    "- All 11 rounds (2-12) completed successfully.\n"
    if not failed_rounds
    else "- **Rounds that FAILED and must be rerun:** "
         + "; ".join(f"round {r} ({msg})" for r, msg in failed_rounds) + "\n"
)

report = f"""# Chronos-T5 Base (200M) - M6 Rounds 2-12 Inference Report

Generated: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}
Run started: {run_started_utc.strftime('%Y-%m-%d %H:%M:%S UTC')}

## Experiment settings (identical for every round)

| Setting | Value |
|---|---|
| Model | `{MODEL_ID}` (Chronos-T5 Base, 200M parameters) |
| Context length | {CONTEXT_LENGTH} weekday daily log returns |
| Prediction length | {PREDICTION_LENGTH} weekdays |
| Sampled trajectories | {NUM_SAMPLES} per asset |
| Series batch size | {SERIES_BATCH_SIZE} (GPU efficiency only) |
| Random seed | {RANDOM_SEED} (reset at the start of every round) |
| Assets | {N_ASSETS} official M6 assets, official order |
| Runtime | Google Colab GPU ({GPU_NAME}), dtype {DTYPE} |
| Versions | chronos-forecasting {version('chronos-forecasting')}, torch {torch.__version__} |

- The model was **loaded once** before the round loop and reused for every round;
  it was never reloaded, trained or fine-tuned (`MODEL_LOADED_ONCE = {MODEL_LOADED_ONCE}`).
- Chronos is univariate: each of the {N_ASSETS} assets was forecast as its own
  independent series. Batching of {SERIES_BATCH_SIZE} assets per forward pass is a
  speed/memory measure only and cannot let one asset's history inform another's forecast.
- Contexts were taken unchanged from the Stage 3 files
  `round_XX_context.csv`, each verified byte-for-byte against the repository
  SHA-256 digests before inference. No preprocessing was repeated: no normalising,
  standardising, smoothing, clipping, averaging, outlier removal or NaN-filling.
  OGN's genuine leading missing values were left as NaN and handled by Chronos's
  missing-value mask.

## Per-round results

| Round | Origin (context end) | Forecast range | Output shape | Finite check | NPZ file |
|---|---|---|---|---|---|
{table}

{failure_note}
Each NPZ contains exactly `forecast_samples` (assets x sampled trajectories x
forecast weekdays), `asset_symbols` and `forecast_dates`. Every file was saved
immediately after its round finished, then reloaded and confirmed to open with the
correct shape, unchanged asset ordering, correct forecast dates and finite values.

## Context provenance (SHA-256)

{chr(10).join(f"- Round {rec['round']:02d}: `{rec['context_sha256']}`" for rec in round_records)}

## Scope and confirmations

- Raw sampled trajectories were saved exactly as produced by the model. **No
  post-processing** of any kind was applied: no averaging or aggregation of the
  trajectories, no four-week return conversion, no rescaling.
- **No quintile probabilities and no RPS evaluation** occurred in this notebook;
  no realised/ground-truth future returns were loaded.
- No asset ranking, no forecast-accuracy evaluation.
- **DRE:** raw Chronos output was preserved for every round, including the
  post-acquisition rounds (9-12) where non-zero values may appear. DRE forecasts
  were **not** overwritten or forced to zero here, and PLD was not used. The
  official M6 zero-return rule for DRE will be applied later, during the
  competition-specific post-processing/evaluation stage.
"""

REPORT_PATH.write_text(report, encoding="utf-8")
print(f"Report written to: {REPORT_PATH}")
print("Copy it into the repository as "
      "Results/Chronos_T5_Base_200M/reports/chronos_t5_base_rounds02_12_inference_report.md")

Report written to: /content/drive/MyDrive/HonoursResearch/outputs/chronos_t5_base/chronos_t5_base_rounds02_12_inference_report.md
Copy it into the repository as Results/Chronos_T5_Base_200M/reports/chronos_t5_base_rounds02_12_inference_report.md


## 10. Completion summary

A final one-screen confirmation of what this run produced, and what to copy back
into the repository.

In [12]:
print("Chronos-T5 Base (200M) - M6 Rounds 2-12")
print(f"  model             : {MODEL_ID} (loaded once, inference only)")
print(f"  rounds completed  : {len(round_records)} / {len(list(ROUNDS))} "
      f"-> {[rec['round'] for rec in round_records]}")
if failed_rounds:
    print(f"  rounds FAILED     : {[r for r, _ in failed_rounds]}")
print(f"  per-round shape   : ({N_ASSETS}, {NUM_SAMPLES}, {PREDICTION_LENGTH}) "
      "(assets x trajectories x forecast weekdays)")
print(f"  seed              : {RANDOM_SEED} (reset each round)")
print(f"  saved to          : {DRIVE_OUTPUT_DIR}")
print(f"  all files verified: {ALL_ROUNDS_VERIFIED}")
print(f"  report            : {REPORT_PATH.name}")
print("  post-processing   : none (no four-week returns, quintiles, or RPS)")
print("  DRE               : raw forecasts preserved, not zeroed (rule applied later)")
print("\nCopy back into the repository:")
print("  NPZ files -> Results/Chronos_T5_Base_200M/round_outputs/")
print("  report    -> Results/Chronos_T5_Base_200M/reports/")

Chronos-T5 Base (200M) - M6 Rounds 2-12
  model             : amazon/chronos-t5-base (loaded once, inference only)
  rounds completed  : 11 / 11 -> [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
  per-round shape   : (100, 100, 20) (assets x trajectories x forecast weekdays)
  seed              : 42 (reset each round)
  saved to          : /content/drive/MyDrive/HonoursResearch/outputs/chronos_t5_base
  all files verified: True
  report            : chronos_t5_base_rounds02_12_inference_report.md
  post-processing   : none (no four-week returns, quintiles, or RPS)
  DRE               : raw forecasts preserved, not zeroed (rule applied later)

Copy back into the repository:
  NPZ files -> Results/Chronos_T5_Base_200M/round_outputs/
  report    -> Results/Chronos_T5_Base_200M/reports/
